In [1]:
##import libs and load the model
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

##model = load_model('sentiment_analysis_model.h5')

In [2]:
##Load the IMDB dataset word index
word_index=imdb.get_word_index()
reverse_word_index=dict([(value,key) for (key,value) in word_index.items()])

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
model=load_model('/content/drive/MyDrive/Colab Notebooks/imdb_lstm_rnn.h5')

In [5]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (32, 500, 128)         │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (32, 64)               │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (32, 1)                │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,329,475 (5.07 MB)

 Trainable params: 1,329,473 (5.07 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [6]:
model.get_weights()

[array([[ 0.21048015, -0.13591619, -0.12468126, ...,  0.29840598,
          0.10825198, -0.06522249],
        [ 0.07870746, -0.06246875, -0.10615847, ...,  0.12708196,
         -0.01989886, -0.09291977],
        [ 0.18399265,  0.11946368,  0.02248504, ...,  0.03006453,
          0.04624308, -0.07366948],
        ...,
        [ 0.00533615, -0.08893977, -0.06359958, ...,  0.00999339,
         -0.0348548 , -0.0037461 ],
        [-0.07952251,  0.06190049,  0.03400552, ..., -0.06947231,
          0.14143606,  0.07463449],
        [-0.1962944 , -0.12452266, -0.17549227, ..., -0.17491247,
         -0.1343548 , -0.12232749]], dtype=float32),
 array([[-0.38963845, -0.30246174, -0.3011948 , ..., -0.05109616,
          0.04539288, -0.09983182],
        [-0.10635468, -0.13126737, -0.13456826, ..., -0.19258979,
          0.20751621, -0.06466573],
        [ 0.06482771, -0.03591309,  0.06915512, ..., -0.05837568,
         -0.02698226, -0.12106487],
        ...,
        [-0.16264236, -0.2280151 , -0.2

In [7]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model
import re

# 1. Load the model (Make sure 'simple_rnn_imdb.h5' is uploaded to your Colab files sidebar)
#model = load_model('simple_rnn_imdb.h5')

# 2. Re-download clean dictionary maps from Keras source
raw_word_index = imdb.get_word_index()
word_index = {k: v for k, v in raw_word_index.items()}
reverse_word_index = {v + 3: k for k, v in raw_word_index.items()}
reverse_word_index[0] = '<PAD>'
reverse_word_index[1] = '<START>'
reverse_word_index[2] = '<UNK>'
reverse_word_index[3] = '<UNUSED>'

# 3. Helper Functions
def preprocess_text(text):
    text = re.sub(r'[^\w\s]', '', text) # Strip out punctuation (. or !)
    words = text.lower().split()

    encoded_review = [1]  # Start sequence token added correctly here
    for word in words:
        if word in word_index:
            index = word_index[word] + 3
            if index < 10000: # Adjust if your training vocabulary was different
                encoded_review.append(index)
            else:
                encoded_review.append(2)
        else:
            encoded_review.append(2)

    return sequence.pad_sequences([encoded_review], maxlen=500)

def decode_review(encoded_review):
    # Flatten array for safe parsing
    flat_review = encoded_review[0] if len(encoded_review.shape) > 1 else encoded_review
    return ' '.join([reverse_word_index.get(i, '?') for i in flat_review if i != 0])

# 4. Test execution
test_review = "This movie was absolutely terrible and awful. I hated it completely."
processed = preprocess_text(test_review)

# Diagnostic Printout: Let's see what the model is actually reading!
print("--- 🔍 COLAB DIAGNOSTIC CHECK ---")
print(f"What the model reads: {decode_review(processed)}\n")

# Run prediction
raw_score = model.predict(processed)
print(f"Raw Model Score: {raw_score[0][0]}")
print(f"Final Sentiment: {'Positive' if raw_score[0][0] > 0.5 else 'Negative'}")


--- 🔍 COLAB DIAGNOSTIC CHECK ---
What the model reads: <START> this movie was absolutely terrible and awful i hated it completely

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 499ms/step
Raw Model Score: 0.01839476265013218
Final Sentiment: Negative


In [8]:
### Prediction  function

def predict_sentiment(review):
    preprocessed_input=preprocess_text(review)

    prediction=model.predict(preprocessed_input)

    sentiment = 'Positive' if prediction[0] [0]> 0.5 else 'Negative'

    return sentiment, prediction[0][0]

In [15]:
##Example review for prediction
example_review="I didn't hate the movie"
sentiment,score=predict_sentiment(example_review)
print(f'Sentiment: {sentiment}, Score: {score}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
Sentiment: Negative, Score: 0.12905116379261017
